# Analyse d'erreurs et explicabilité d'un classifieur d'images

Un classifieur de photos de paysages (6 classes : bâtiments, forêt, glacier, montagne, mer, rue) se trompe sur environ une image sur dix. Sur quelles images, pourquoi, et faut-il changer de modèle ou corriger les données ? Les images sont celles du jeu [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification), des photos de 150 × 150 pixels.

1. Un modèle pré-entraîné
2. L'entraînement de sa dernière couche
3. Les tranches à travailler d'abord
4. La matrice de confusion et les causes des erreurs
5. Expliquer des erreurs avec SHAP
6. Modèle ou étiquettes ?

Le notebook tourne sans GPU, en une dizaine de minutes : le réseau pré-entraîné reste gelé, ses sorties sont calculées une seule fois, sur un sous-ensemble des images d'entraînement.

## Données

Les images sont rangées dans un dossier par classe : `landscape/seg_train` (14 034 images) et `landscape/seg_test` (3 000). `fabrique_chargeur` les charge avec les transformations du modèle pré-entraîné, `POIDS.transforms()` (redimensionnement en 224 × 224 et normalisation d'ImageNet). Pour tenir en quelques minutes sans GPU, le modèle ne s'entraîne que sur `NB_PAR_CLASSE` images par classe, tirées au hasard ; le jeu de test est complet.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL.Image
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
POIDS = models.MobileNet_V3_Small_Weights.DEFAULT
NB_PAR_CLASSE = 300  # images d'entraînement par classe


def fabrique_chargeur(
  dossier: str, melange: bool = False, par_classe: int | None = None
) -> DataLoader:
  dataset = datasets.ImageFolder(dossier, transform=POIDS.transforms())
  if par_classe is not None:  # par_classe images tirées au hasard dans chaque classe
    random.seed(0)
    dataset.samples = [
      echantillon
      for cible in range(len(dataset.classes))
      for echantillon in random.sample(
        [s for s in dataset.samples if s[1] == cible], par_classe
      )
    ]
    dataset.targets = [cible for _, cible in dataset.samples]
  return DataLoader(dataset, batch_size=64, shuffle=melange)


chargeur_images_entrainement = fabrique_chargeur(
  "landscape/seg_train", par_classe=NB_PAR_CLASSE
)
chargeur_images_test = fabrique_chargeur("landscape/seg_test")
CLASSES = chargeur_images_test.dataset.classes
chemins_entrainement = [
  chemin for chemin, _ in chargeur_images_entrainement.dataset.samples
]
chemins_test = [chemin for chemin, _ in chargeur_images_test.dataset.samples]
print(f"Classes : {CLASSES}")
print(f"{len(chemins_entrainement)} images d'entraînement, {len(chemins_test)} de test")

## 1. Un modèle pré-entraîné

On part d'un MobileNetV3 *small* pré-entraîné sur ImageNet (les poids `POIDS`). Sa partie convolutive, `modele.features`, est gelée (`requires_grad_(False)` : plus de gradient pour ses paramètres), et sa dernière couche, `modele.classifier[-1]`, qui sortait les 1 000 classes d'ImageNet, est remplacée par une couche linéaire à une sortie par classe de `CLASSES`. Seules les couches de `modele.classifier` restent entraînables.

In [ ]:
modele = models.mobilenet_v3_small(weights=POIDS)
modele.features.requires_grad_(False)
modele.classifier[-1] = nn.Linear(modele.classifier[-1].in_features, len(CLASSES))
modele = modele.to(DEVICE)

entrainables = sum(p.numel() for p in modele.parameters() if p.requires_grad)
total = sum(p.numel() for p in modele.parameters())
print(f"{entrainables:,} paramètres entraînables sur {total:,}")

## 2. L'entraînement de la dernière couche

La partie gelée renvoie toujours le même vecteur de 576 caractéristiques pour une image donnée : `extrait_caracteristiques` le calcule une fois pour toutes, pour chaque image (quelques minutes sans GPU). Les couches de `modele.classifier` s'entraînent ensuite sur ces vecteurs, en quelques secondes, avec les fonctions habituelles `entraine_epoque`, `evalue` et `predit`.

In [ ]:
@torch.inference_mode()
def extrait_caracteristiques(modele: nn.Module, chargeur: DataLoader) -> TensorDataset:
  """Les sorties de la partie gelée (features puis avgpool), et les étiquettes."""
  modele.eval()
  caracteristiques, etiquettes = [], []
  for images, cibles in chargeur:
    caracteristiques.append(
      modele.avgpool(modele.features(images.to(DEVICE))).flatten(1).cpu()
    )
    etiquettes.append(cibles)
  return TensorDataset(torch.cat(caracteristiques), torch.cat(etiquettes))


caracteristiques_entrainement = extrait_caracteristiques(
  modele, chargeur_images_entrainement
)
caracteristiques_test = extrait_caracteristiques(modele, chargeur_images_test)
print(f"Caractéristiques de test : {tuple(caracteristiques_test.tensors[0].shape)}")

In [ ]:
def entraine_epoque(modele: nn.Module, chargeur: DataLoader, optimiseur) -> float:
  """Une époque ; renvoie la perte moyenne."""
  modele.train()
  fonction_perte = nn.CrossEntropyLoss()
  perte_totale = 0.0
  for x, y in chargeur:
    x, y = x.to(DEVICE), y.to(DEVICE)
    optimiseur.zero_grad()
    perte = fonction_perte(modele(x), y)
    perte.backward()
    optimiseur.step()
    perte_totale += perte.item() * len(x)
  return perte_totale / len(chargeur.dataset)


@torch.inference_mode()
def evalue(modele: nn.Module, chargeur: DataLoader) -> float:
  """Exactitude (accuracy) sur chargeur."""
  modele.eval()
  corrects = 0
  for x, y in chargeur:
    corrects += (modele(x.to(DEVICE)).argmax(dim=1).cpu() == y).sum().item()
  return corrects / len(chargeur.dataset)


@torch.inference_mode()
def predit(modele: nn.Module, chargeur: DataLoader) -> torch.Tensor:
  """Indices des classes prédites."""
  modele.eval()
  return torch.cat([modele(x.to(DEVICE)).argmax(dim=1).cpu() for x, _ in chargeur])


torch.manual_seed(0)
chargeur_entrainement = DataLoader(
  caracteristiques_entrainement, batch_size=64, shuffle=True
)
chargeur_test = DataLoader(caracteristiques_test, batch_size=64)
optimiseur = torch.optim.AdamW(modele.classifier.parameters(), lr=0.001)
for epoque in range(10):
  perte = entraine_epoque(modele.classifier, chargeur_entrainement, optimiseur)
  print(f"Époque {epoque + 1} : perte {perte:.3f}")
print(f"Exactitude sur le jeu de test : {evalue(modele.classifier, chargeur_test):.3f}")

## 3. Les tranches à travailler d'abord

Les classes sont des tranches naturelles des données : la méthode de la frame « Approche conseillée — Exemple, fin » dit laquelle travailler d'abord. Calculez `tranches`, un `DataFrame` indexé par `CLASSES`, avec, pour chaque classe, son exactitude sur les images de test (la part de ses images bien classées) et sa proportion parmi les images de test. `REFERENCE` donne l'exactitude par classe d'un modèle plus fort, le même réseau entraîné sur les 14 034 images d'entraînement (la démo de ce TP) : faute de performance humaine mesurée, c'est la référence de ce qui est atteignable. Ajoutez l'impact potentiel de chaque classe, (référence − exactitude) × proportion, et classez les classes par impact décroissant. Quelle tranche travailler d'abord ? Le classement par impact est-il celui de l'exactitude seule ?

**Documentation utile**

- [`numpy.bincount`](https://numpy.org/doc/stable/reference/generated/numpy.bincount.html)
- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`DataFrame.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)

In [ ]:
etiquettes_test = caracteristiques_test.tensors[1].numpy()
predictions_test = predit(modele.classifier, chargeur_test).numpy()
# Exactitude par classe du même réseau entraîné sur les 14 034 images
REFERENCE = {
  "buildings": 0.957,
  "forest": 0.996,
  "glacier": 0.899,
  "mountain": 0.838,
  "sea": 0.967,
  "street": 0.902,
}

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `etiquettes_test` et `predictions_test` : les vraies classes des images de test et les classes prédites, dans l'ordre de `chemins_test`
- `(predictions_test[etiquettes_test == c] == c).mean()` : l'exactitude de la classe d'indice `c`, la part de ses images prédites `c`
- `np.bincount(etiquettes_test) / len(etiquettes_test)` : la proportion de chaque classe, dans l'ordre de `CLASSES`
- `pd.DataFrame({"exactitude": ..., "proportion": ...}, index=CLASSES)` : une ligne par classe ; `pd.Series(REFERENCE)`, ajoutée en colonne, s'aligne sur cet index
- `tranches.sort_values("impact", ascending=False)` : les plus forts impacts d'abord
- Pour vérifier : les proportions somment à 1 ; un impact négatif veut dire que le modèle fait mieux que la référence sur cette classe

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
exactitudes = [
  (predictions_test[etiquettes_test == c] == c).mean() for c in range(len(CLASSES))
]
tranches = pd.DataFrame(
  {"exactitude": ..., "proportion": np.bincount(...) / len(etiquettes_test)},
  index=CLASSES,
)
tranches["référence"] = pd.Series(REFERENCE)
tranches["impact"] = (... - tranches["exactitude"]) * ...
tranches.sort_values(..., ascending=False).round(3)
```

### Solution

In [ ]:
exactitudes = [
  (predictions_test[etiquettes_test == c] == c).mean() for c in range(len(CLASSES))
]
tranches = pd.DataFrame(
  {
    "exactitude": exactitudes,
    "proportion": np.bincount(etiquettes_test) / len(etiquettes_test),
  },
  index=CLASSES,
)
tranches["référence"] = pd.Series(REFERENCE)
tranches["impact"] = (tranches["référence"] - tranches["exactitude"]) * tranches[
  "proportion"
]
tranches.sort_values("impact", ascending=False).round(3)

Éléments de réponse :

- Sur notre exécution, « glacier » a le plus fort impact (1,4 % des images de test) : c'est la tranche à travailler d'abord. Viennent ensuite « sea » et « buildings », loin derrière.
- Le classement n'est pas celui de l'exactitude : « mountain » a la deuxième exactitude la plus basse (0,87), mais un impact négatif. Le modèle plus fort ne fait pas mieux sur cette classe : il n'y a rien à y gagner en l'imitant. L'exactitude dit où le modèle se trompe, l'impact où il peut progresser.
- « glacier » et « mountain » se lisent ensemble : ce modèle range plus d'images en « mountain » que le modèle de référence, ce qui fait monter l'exactitude de « mountain » et baisser celle de « glacier ». La matrice de confusion, dans la partie suivante, le montre.
- La référence n'est pas une performance humaine : sur cette paire, un humain hésiterait aussi (parties 5 et 6), et le gain atteignable est peut-être plus faible.
- Vos chiffres peuvent différer un peu : sans GPU non plus, l'entraînement ne donne pas exactement le même modèle d'une exécution à l'autre.

## 4. La matrice de confusion et les causes des erreurs

Affichez la matrice de confusion des images de test, à partir de `etiquettes_test` et `predictions_test`. Calculez ensuite `paires`, les trois paires (vraie classe, classe prédite) les plus fréquentes parmi les erreurs, avec leur nombre d'images. Affichez enfin, avec `affiche_images`, une dizaine d'images de la paire la plus fréquente. Ces erreurs se ressemblent-elles ? Les deux classes de la paire se distinguent-elles toujours à l'œil ?

**Documentation utile**

- [`ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)
- [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [`numpy.flatnonzero`](https://numpy.org/doc/stable/reference/generated/numpy.flatnonzero.html)

In [ ]:
def affiche_images(chemins: list[str], titres: list[str]) -> None:
  """Affiche des images côte à côte, avec leurs titres."""
  _, axes = plt.subplots(
    1, len(chemins), figsize=(2 * len(chemins), 2.4), squeeze=False
  )
  for ax, chemin, titre in zip(axes[0], chemins, titres):
    ax.imshow(PIL.Image.open(chemin))
    ax.set_title(titre, fontsize=9)
    ax.axis("off")
  plt.show()


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `ConfusionMatrixDisplay.from_predictions(etiquettes_test, predictions_test, display_labels=CLASSES, xticks_rotation=45)` : les vraies classes en lignes, les prédites en colonnes
- `confusion_matrix(etiquettes_test, predictions_test)[vraie, predite]` : le nombre d'images de la classe `vraie` prédites `predite` ; les erreurs sont hors de la diagonale (`vraie != predite`)
- `sorted(..., reverse=True)[:3]` sur des triplets `(nombre, vraie classe, classe prédite)` : les trois plus grands nombres d'abord
- `np.flatnonzero((etiquettes_test == a) & (predictions_test == b))` : les positions des images de la classe `a` prédites `b`, qui sont aussi leurs positions dans `chemins_test`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

ConfusionMatrixDisplay.from_predictions(
  ..., ..., display_labels=CLASSES, xticks_rotation=45
)
plt.show()

matrice = confusion_matrix(etiquettes_test, predictions_test)
paires = sorted(
  (
    (matrice[vraie, predite], CLASSES[vraie], CLASSES[predite])
    for vraie in range(len(CLASSES))
    for predite in range(len(CLASSES))
    if ...
  ),
  reverse=True,
)[:3]
for effectif, vraie, predite in paires:
  print(f"{effectif} images « {vraie} » prédites « {predite} »")

effectif, vraie, predite = paires[0]
erreurs = np.flatnonzero(
  (etiquettes_test == CLASSES.index(vraie)) & (... == CLASSES.index(...))
)
affiche_images(
  [chemins_test[i] for i in erreurs[:10]], [f"n° {i}" for i in erreurs[:10]]
)
```

### Solution

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

ConfusionMatrixDisplay.from_predictions(
  etiquettes_test, predictions_test, display_labels=CLASSES, xticks_rotation=45
)
plt.show()

matrice = confusion_matrix(etiquettes_test, predictions_test)
paires = sorted(
  (
    (matrice[vraie, predite], CLASSES[vraie], CLASSES[predite])
    for vraie in range(len(CLASSES))
    for predite in range(len(CLASSES))
    if vraie != predite
  ),
  reverse=True,
)[:3]
for effectif, vraie, predite in paires:
  print(f"{effectif} images « {vraie} » prédites « {predite} »")

effectif, vraie, predite = paires[0]
erreurs = np.flatnonzero(
  (etiquettes_test == CLASSES.index(vraie))
  & (predictions_test == CLASSES.index(predite))
)
affiche_images(
  [chemins_test[i] for i in erreurs[:10]], [f"n° {i}" for i in erreurs[:10]]
)

### Les causes des erreurs

La matrice dit quelles classes se confondent, pas pourquoi. Comme dans la frame « Trouver les tranches », examinez les 20 premières erreurs de la paire la plus fréquente, `examinees`, et donnez à chacune une étiquette de cause : « montagne enneigée » (des sommets et de la neige ou de la glace à la fois : un cas ambigu, où un humain hésiterait aussi), « étiquette fausse » (l'image n'est pas de sa classe : pas de glace sur un « glacier », par exemple), « image atypique » (une vue aérienne, un dessin, un cadrage inhabituel), ou « autre » (une image claire de sa classe, que le modèle manque). Remplissez `causes`, un dictionnaire {numéro d'image : étiquette de cause}, puis comptez les erreurs par étiquette de cause avec un `Counter`. Quelle cause domine ? Quelle tranche mesurer ensuite, et que faire pour les erreurs de chaque cause ?

**Documentation utile**

- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)

In [ ]:
from collections import Counter

CAUSES = ["montagne enneigée", "étiquette fausse", "image atypique", "autre"]
examinees = erreurs[:20]
for ligne in (examinees[:10], examinees[10:]):
  affiche_images([chemins_test[i] for i in ligne], [f"n° {i}" for i in ligne])

causes = {}  # {numéro d'image : l'une des CAUSES}

# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- Les clés de `causes` sont les numéros affichés au-dessus des images, les valeurs des chaînes de `CAUSES` : `{924: "montagne enneigée", 966: "autre", ...}`
- Une seule étiquette par image, la cause principale ; la frame en admet plusieurs par erreur : un dictionnaire de listes et un `Counter` sur toutes les étiquettes le permettraient
- `Counter(causes.values())` compte les étiquettes ; `.most_common()` les renvoie de la plus fréquente à la moins fréquente, en paires (étiquette, nombre)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
causes = {
  ...: "montagne enneigée",
  ...: "étiquette fausse",
  # ...
}
effectifs = Counter(...)
for cause, effectif in effectifs.most_common():
  print(f"{cause} : {effectif} ({effectif / len(causes):.0%})")
```

#### Solution

In [ ]:
causes = {
  922: "image atypique",
  924: "montagne enneigée",
  938: "étiquette fausse",
  943: "montagne enneigée",
  950: "étiquette fausse",
  956: "étiquette fausse",
  962: "montagne enneigée",
  966: "autre",
  976: "montagne enneigée",
  994: "montagne enneigée",
  999: "montagne enneigée",
  1001: "autre",
  1020: "autre",
  1030: "autre",
  1033: "image atypique",
  1034: "image atypique",
  1043: "image atypique",
  1049: "étiquette fausse",
  1050: "autre",
  1054: "montagne enneigée",
}
effectifs = Counter(causes.values())
for cause, effectif in effectifs.most_common():
  print(f"{cause} : {effectif} ({effectif / len(causes):.0%})")

Éléments de réponse (sur notre exécution, des glaciers prédits « mountain » ; vos erreurs peuvent différer un peu) :

- « montagne enneigée » domine (7 sur 20, 35 %) : des sommets et de la glace dans la même image, que l'étiquette soit « glacier » ou « mountain ». C'est la tranche « Montagnes enneigées » de la frame « Approche conseillée — Exemple » : on la mesure ensuite à part (l'exactitude du modèle, et la performance humaine sur un échantillon ré-annoté), et on la traite par le guide d'annotation, pas par le modèle.
- « autre » (5) : des glaciers sans ambiguïté (séracs, fronts de glace) que le modèle manque. Là, c'est le modèle qui se trompe : plus d'images d'entraînement (le modèle de référence, entraîné sur toutes les images, fait mieux sur « glacier »), ou de l'augmentation de données.
- « étiquette fausse » (4) : des lacs, une forêt, des montagnes vertes, étiquetés « glacier ». On les ré-annote ; aucun modèle ne peut les « réussir ».
- « image atypique » (4) : des vues aériennes, une plage d'icebergs, rares dans les données. On en ajoute de semblables, ou on accepte ces erreurs en connaissance de cause.
- Vingt erreurs donnent un ordre de grandeur ; la frame conseille d'en examiner une centaine avant de décider.

## 5. Expliquer des erreurs avec SHAP

Les montagnes prises pour des glaciers comptent parmi les erreurs les plus fréquentes. Pour voir ce qui pousse le modèle vers « glacier », expliquez-en six avec SHAP, celles de `images`. Écrivez d'abord `predit_probabilites(images)`, qui prend un lot d'images NumPy de forme (N, 150, 150, 3), aux valeurs entre 0 et 1, et renvoie les probabilités des 6 classes, de forme (N, 6) : SHAP n'appelle le modèle qu'à travers cette fonction. Créez ensuite un explainer qui masque des zones des images en les remplaçant par du gris, calculez `explication` pour les sorties « glacier » et « mountain », et affichez-la. Quelles zones des images poussent le modèle vers « glacier » ? Vers « mountain » ?

**Documentation utile**

- [`Tensor.permute`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.permute.html)
- [`shap.Explainer`](https://shap.readthedocs.io/en/latest/generated/shap.Explainer.html)
- [`shap.maskers.Image`](https://shap.readthedocs.io/en/latest/generated/shap.maskers.Image.html)
- [`shap.plots.image`](https://shap.readthedocs.io/en/latest/generated/shap.plots.image.html)

In [ ]:
GLACIER, MONTAGNE = CLASSES.index("glacier"), CLASSES.index("mountain")
montagnes_predites_glacier = np.flatnonzero(
  (etiquettes_test == MONTAGNE) & (predictions_test == GLACIER)
)
print(f"{len(montagnes_predites_glacier)} montagnes prédites glacier")


def charge_image(chemin: str) -> np.ndarray:
  """Une image en tableau (150, 150, 3), aux valeurs entre 0 et 1."""
  image = PIL.Image.open(chemin).convert("RGB").resize((150, 150))
  return np.asarray(image, dtype=np.float32) / 255


images = np.stack(
  [charge_image(chemins_test[i]) for i in montagnes_predites_glacier[:6]]
)


def predit_probabilites(images: np.ndarray) -> np.ndarray: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.as_tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)` : d'un lot NumPy (N, hauteur, largeur, canaux) à un tenseur PyTorch (N, canaux, hauteur, largeur)
- `POIDS.transforms()(lot)` : le même redimensionnement et la même normalisation que pendant l'entraînement
- `modele(...).softmax(dim=1)`, sous `torch.inference_mode()` après `modele.eval()`, puis `.cpu().numpy()`
- Pour vérifier : chaque ligne de `predit_probabilites(images)` a sa plus grande probabilité dans la colonne `GLACIER` (2), et sa somme vaut 1
- `shap.maskers.Image(0.5, images[0].shape)` : les zones masquées sont remplacées par du gris (0,5) ; le masqueur `"blur(32,32)"` des exemples de shap demande OpenCV
- `shap.Explainer(predit_probabilites, masque, output_names=CLASSES)` : avec un masqueur d'images, shap choisit l'explainer *Partition*, qui découpe l'image en blocs de plus en plus petits
- `explicateur(images, max_evals=500, batch_size=50, outputs=[GLACIER, MONTAGNE])` : 500 appels au modèle par image (une quinzaine de secondes par image sans GPU), par lots de 50 ; `outputs` : les classes à expliquer
- `shap.plots.image(explication)` : en rouge, les zones qui poussent vers la classe de la colonne ; en bleu, celles qui en éloignent

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import shap


@torch.inference_mode()
def predit_probabilites(images: np.ndarray) -> np.ndarray:
  modele.eval()
  lot = torch.as_tensor(images, dtype=torch.float32).permute(...)
  return modele(POIDS.transforms()(lot).to(DEVICE)).softmax(dim=...).cpu().numpy()


print(predit_probabilites(images).round(2))

masque = shap.maskers.Image(0.5, images[0].shape)
explicateur = shap.Explainer(..., masque, output_names=CLASSES)
explication = explicateur(images, max_evals=500, batch_size=50, outputs=[..., ...])
shap.plots.image(...)
```

### Solution

In [ ]:
import shap


@torch.inference_mode()
def predit_probabilites(images: np.ndarray) -> np.ndarray:
  modele.eval()
  lot = torch.as_tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)
  return modele(POIDS.transforms()(lot).to(DEVICE)).softmax(dim=1).cpu().numpy()


print(predit_probabilites(images).round(2))

masque = shap.maskers.Image(0.5, images[0].shape)
explicateur = shap.Explainer(predit_probabilites, masque, output_names=CLASSES)
explication = explicateur(
  images, max_evals=500, batch_size=50, outputs=[GLACIER, MONTAGNE]
)
shap.plots.image(explication)

## 6. Modèle ou étiquettes ?

Une erreur vient du modèle quand l'image est claire, mais qu'il s'appuie sur un mauvais indice. Elle vient des étiquettes quand des images semblables portent tantôt une étiquette, tantôt l'autre : aucun modèle ne peut alors apprendre la frontière. Pour trancher, écrivez `cherche_voisines(indices, k)`, qui renvoie, pour chaque image de test de `indices`, les positions dans `chemins_entrainement` des `k` images d'entraînement les plus proches, dans l'espace des caractéristiques du réseau gelé (les vecteurs de `caracteristiques_test` et de `caracteristiques_entrainement`, comparés par similarité cosinus). Comparez la part d'étiquettes « glacier » parmi les 5 voisines des montagnes prédites glacier, et parmi celles des montagnes bien classées. Affichez enfin les trois premières erreurs de `montagnes_predites_glacier` et leurs voisines, avec leurs étiquettes. D'après ces voisines et les figures de SHAP, ces erreurs viennent-elles plutôt du modèle ou des étiquettes ? Que faudrait-il faire ?

**Documentation utile**

- [`torch.nn.functional.normalize`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html)
- [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)

In [ ]:
def cherche_voisines(indices: np.ndarray, k: int = 5) -> np.ndarray: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `normalize(vecteurs, dim=1)` (de `torch.nn.functional`) : chaque vecteur ramené à une norme de 1 ; le produit scalaire de deux vecteurs normalisés est leur similarité cosinus
- `vecteurs_test[indices] @ vecteurs_entrainement.T` : la similarité de chaque image de test de `indices` avec chaque image d'entraînement, de forme (len(indices), 1 800)
- `similarites.topk(k, dim=1).indices` : les positions des `k` plus grandes similarités de chaque ligne ; ce sont des positions dans `chemins_entrainement` et dans `caracteristiques_entrainement`
- `etiquettes_entrainement[cherche_voisines(indices)]` : les étiquettes des voisines, de forme (len(indices), k) ; `(... == GLACIER).mean()` : la part de « glacier »
- Pour vérifier : `cherche_voisines(montagnes_predites_glacier[:2]).shape` vaut `(2, 5)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from torch.nn.functional import normalize

vecteurs_entrainement = normalize(caracteristiques_entrainement.tensors[0], dim=1)
vecteurs_test = normalize(..., dim=1)
etiquettes_entrainement = caracteristiques_entrainement.tensors[1].numpy()


def cherche_voisines(indices: np.ndarray, k: int = 5) -> np.ndarray:
  similarites = vecteurs_test[indices] @ vecteurs_entrainement.T
  return similarites.topk(..., dim=1).indices.numpy()


bien_classees = np.flatnonzero(
  (etiquettes_test == MONTAGNE) & (predictions_test == ...)
)
for nom, indices in [
  ("Montagnes prédites glacier", montagnes_predites_glacier),
  ("Montagnes bien classées", bien_classees),
]:
  part = (etiquettes_entrainement[cherche_voisines(indices)] == ...).mean()
  print(f"{nom} : {part:.0%} de voisines étiquetées « glacier »")

for i, proches in zip(
  montagnes_predites_glacier[:3], cherche_voisines(montagnes_predites_glacier[:3])
):
  affiche_images(
    [chemins_test[i]] + [chemins_entrainement[j] for j in proches],
    [f"test n° {i}"] + [CLASSES[...] for j in proches],
  )
```

### Solution

In [ ]:
from torch.nn.functional import normalize

vecteurs_entrainement = normalize(caracteristiques_entrainement.tensors[0], dim=1)
vecteurs_test = normalize(caracteristiques_test.tensors[0], dim=1)
etiquettes_entrainement = caracteristiques_entrainement.tensors[1].numpy()


def cherche_voisines(indices: np.ndarray, k: int = 5) -> np.ndarray:
  similarites = vecteurs_test[indices] @ vecteurs_entrainement.T
  return similarites.topk(k, dim=1).indices.numpy()


bien_classees = np.flatnonzero(
  (etiquettes_test == MONTAGNE) & (predictions_test == MONTAGNE)
)
for nom, indices in [
  ("Montagnes prédites glacier", montagnes_predites_glacier),
  ("Montagnes bien classées", bien_classees),
]:
  part = (etiquettes_entrainement[cherche_voisines(indices)] == GLACIER).mean()
  print(f"{nom} : {part:.0%} de voisines étiquetées « glacier »")

for i, proches in zip(
  montagnes_predites_glacier[:3], cherche_voisines(montagnes_predites_glacier[:3])
):
  affiche_images(
    [chemins_test[i]] + [chemins_entrainement[j] for j in proches],
    [f"test n° {i}"] + [CLASSES[etiquettes_entrainement[j]] for j in proches],
  )

Éléments de réponse :

- Les montagnes prédites glacier ont bien plus de voisines « glacier » que les montagnes bien classées : dans le jeu d'entraînement, des photos très proches portent l'étiquette « glacier ». Le modèle reproduit l'ambiguïté des étiquettes. La paire inverse, les glaciers prédits montagne, est d'ailleurs, elle aussi, l'une des erreurs les plus fréquentes.
- Les figures de SHAP vont dans le même sens : le rouge de « glacier » tombe sur la neige et la glace (sommets enneigés, champs de neige, fronts de glace), des indices sensés pour cette classe, et non sur un détail sans rapport. Plusieurs de ces « montagnes » montrent d'ailleurs un glacier : un humain hésiterait aussi, et certaines étiquettes sont fausses.
- Ces erreurs viennent donc surtout des étiquettes. On les traite dans les données : préciser le guide d'annotation (par exemple « glacier dès que la glace ou la neige occupe l'essentiel de l'image »), ré-annoter les cas ambigus, ou admettre plusieurs étiquettes par image. Un meilleur modèle n'y gagnerait presque rien : sur cette tranche, la performance humaine de référence est elle-même basse.
- L'erreur viendrait du modèle si ses voisines étaient des montagnes et que SHAP pointait un indice sans rapport (le ciel, un bord de l'image) : on ajouterait alors des exemples plus variés, de l'augmentation de données, ou un meilleur modèle.
- Limites : les blocs de SHAP sont grossiers, et six images ne suffisent pas pour conclure ; on en examine davantage avant de ré-annoter.

## À retenir

- L'analyse d'erreurs commence par la matrice de confusion : les paires de classes les plus confondues, calculées et non supposées, disent quelles erreurs regarder d'abord.
- SHAP explique la prédiction d'une image par zones : il masque des blocs de l'image et mesure l'effet sur la probabilité de chaque classe. Ses blocs sont grossiers, et il explique le modèle, pas la réalité.
- Avant de changer de modèle, regarder les étiquettes : des images semblables étiquetées différemment se corrigent dans le guide d'annotation et par ré-annotation, pas avec un meilleur réseau.